# Atmospheric correction of WorldView-3 imagery

This notebook performs atmospheric correction of WorldView-3 Level-2A imagery using the 6S radiative transfer model.

### Input

- WorldView-3 multispectral image (TIL file)
- metadata (.IMD)

### Output

- atmospherically corrected rasters

In [1]:
from pathlib import Path

# Import the custom function that performs atmospheric correction using the WorldView-3 metadata (.IMD) and the 6S radiative transfer model.

from functions.atmosferic_correction import atmospheric_correction


# Input and output directories
# Update these paths according to your local project structure.

data_dir = Path("../data")
output_dir = Path("../output")
output_dir.mkdir(exist_ok=True)


# WorldView-3 spectral information:
# Center wavelengths (µm) and exoatmospheric solar irradiance (Esun) are taken from the official Maxar (formerly DigitalGlobe)
# WorldView-3 radiometric documentation available in the "documentation" folder

sensor_info = {

    "VNIR": {
        "wavelengths": [
            0.4274, 0.4819, 0.5471, 0.6043,
            0.6601, 0.7227, 0.8240, 0.9136
        ],

        "Esun": [
            1757.89, 2004.61, 1830.18, 1712.07,
            1535.33, 1348.08, 1055.94, 858.77
        ]
    },

    "SWIR": {
        "wavelengths": [
            1.2091, 1.5716, 1.6611, 1.7295,
            2.1637, 2.2022, 2.2593, 2.3292
        ],

        "Esun": [
            479.019, 263.797, 225.283, 197.552,
            90.4178, 85.0642, 76.9507, 68.0988
        ]
    }

}


# Atmospheric correction
# This code searches automatically for all TIL/IMD pairs contained in the VNIR and SWIR folders. Each image is corrected independently.

for sensor, info in sensor_info.items():

    sensor_dir = data_dir / sensor

    til_files = sorted(sensor_dir.glob("*.TIL"))
    imd_files = sorted(sensor_dir.glob("*.IMD"))

    if len(til_files) != len(imd_files):
        raise ValueError(
            f"{sensor}: different number of TIL ({len(til_files)}) "
            f"and IMD ({len(imd_files)}) files."
        )

    for til_file, imd_file in zip(til_files, imd_files):

        output_file = output_dir / f"{til_file.stem}_corrected.tif"

        atmospheric_correction(
            path_til=til_file,
            path_imd=imd_file,
            path_output=output_file,
            wv3_wavelengths=info["wavelengths"],
            Esun=info["Esun"],
            altitude=0.02,
            aot=0.15,
            remove_black_pixels=True,
        )

        print(f"{sensor}: {output_file.name} completed.")